# Chapitre 10 — `map`, `apply` et vectorisation

## Le problème

Toutes les transformations ne sont pas prévues par une méthode Pandas. Classer une commande en « petite », « moyenne », « grosse » ; appliquer une règle métier ; construire une URL par ligne. On est tenté d'écrire une fonction Python et de l'appliquer à chaque valeur : c'est le rôle de `map` et `apply`.

Ces outils sont pratiques, mais ils exécutent du **code Python ligne par ligne**. Ce chapitre montre comment les utiliser correctement, puis comment les remplacer par des opérations **vectorisées**, souvent cent fois plus rapides.

In [ ]:
import numpy as np
import pandas as pd

commandes = pd.read_csv("../donnees/demo/commandes.csv")
produits = pd.read_csv("../donnees/demo/produits.csv")
commandes["montant"] = commandes["quantite"] * commandes["produit_id"].map(produits.set_index("produit_id")["prix"])

## 1. `Series.map` : transformer valeur par valeur

### Avec un dictionnaire ou une Series : une table de correspondance

In [ ]:
libelles = {"web": "Site web", "magasin": "Boutique physique", "telephone": "Vente par téléphone"}
commandes["canal"].map(libelles).head()

Une valeur absente du dictionnaire donne `NaN` :

In [ ]:
pd.Series(["web", "marketplace"]).map(libelles)

### Avec une fonction

In [ ]:
def taille(montant, seuil=100):
    if montant < seuil:
        return "petite"
    elif montant > seuil:
        return "grosse"
    else:
        return "pile"

commandes["montant"].map(taille).value_counts()

### Le piège des valeurs manquantes

> **Pourquoi ?** La fonction suivante classe une valeur par rapport à 0. Que renvoie-t-elle pour `NaN` ?

In [ ]:
def signe(x):
    if x < 0:
        return -1
    elif x > 0:
        return 1
    else:
        return 0

pd.Series([-2.0, 3.0, np.nan]).map(signe)

Toute comparaison avec `NaN` est fausse : `NaN < 0` et `NaN > 0` valent `False`. La fonction tombe dans le `else` et renvoie 0, comme si la valeur était nulle. L'information « manquante » est perdue, sans erreur.

Deux corrections : traiter le cas explicitement, ou demander à `map` de ne pas appeler la fonction sur les valeurs manquantes.

In [ ]:
pd.Series([-2.0, 3.0, np.nan]).map(signe, na_action="ignore")

## 2. `apply` : plus de souplesse

### Sur une Series, avec des arguments

`apply` transmet les arguments supplémentaires à la fonction : par mot-clé, ou via `args=` (un tuple).

In [ ]:
commandes["montant"].apply(taille, seuil=200).value_counts()

In [ ]:
commandes["montant"].apply(taille, args=(50,)).value_counts()

Rappel Python : une fonction peut accepter un nombre variable d'arguments avec `*args` (positionnels, reçus dans un tuple) et `**kwargs` (nommés, reçus dans un dictionnaire). C'est ce mécanisme qui permet à `apply` de faire suivre des arguments qu'il ne connaît pas.

In [ ]:
def montre(*args, **kwargs):
    return f"args={args} kwargs={kwargs}"

montre(1, 2, seuil=200)

### Sur un DataFrame : par colonne ou par ligne

Avec `axis=0` (défaut), la fonction reçoit **chaque colonne** ; avec `axis=1`, **chaque ligne**, sous forme de Series.

In [ ]:
mesures = pd.DataFrame({"capteur_1": [20.1, 21.4, 19.8], "capteur_2": [18.0, 25.3, 19.1]})
mesures.apply(lambda col: col.max() - col.min())             # une valeur par colonne

In [ ]:
mesures.apply(lambda ligne: ligne["capteur_2"] - ligne["capteur_1"], axis=1)   # une valeur par ligne

### Renvoyer plusieurs valeurs : plusieurs colonnes

Si la fonction renvoie une Series, `apply` construit autant de colonnes que la Series a d'éléments.

In [ ]:
def decoupe(produit_id):
    return pd.Series({"prefixe": produit_id[0], "numero": int(produit_id[1:])})

commandes["produit_id"].head(3).apply(decoupe)

## 3. `DataFrame.map` : chaque cellule

Pour appliquer une fonction à **toutes les cellules** d'un DataFrame, on utilise `DataFrame.map`. Cette méthode s'appelait `applymap` avant Pandas 2.1 ; l'ancien nom n'existe plus en Pandas 3.

In [ ]:
mesures.map(lambda v: f"{v:.0f} °C")

## 4. La vectorisation

`map` et `apply` appellent une fonction Python pour chaque valeur. Une opération **vectorisée** traite toute la colonne en une fois, dans du code compilé (C) : c'est ce que vous avez pratiqué avec NumPy.

### Exemple : le signe d'un nombre

La version vectorisée du cours Bstorm utilise l'arithmétique des booléens : `True` vaut 1, `False` vaut 0. Elle produit un code (−1 pour « petite », 0 pour « pile », 1 pour « grosse ») au lieu d'un libellé.

In [ ]:
x = pd.Series(np.random.default_rng(0).uniform(0, 10, 8).round(1))
seuil = 5
vectorise = -1 * (x < seuil) + (x > seuil)
pd.DataFrame({"x": x, "map": x.map(lambda v: taille(v, seuil)), "vectorise": vectorise})

> **Prédiction.** Pour `x = 3`, `-1 * (x < 5) + (x > 5)` vaut `-1 * True + False`. Quelle valeur ? Et pour `x = 5` ?

### `np.where` et `np.select`

`np.where(condition, si_vrai, si_faux)` est le `if/else` vectorisé ; `np.select` gère plusieurs conditions dans l'ordre.

In [ ]:
commandes["gros_panier"] = np.where(commandes["montant"] > 200, "oui", "non")

conditions = [commandes["montant"] < 100, commandes["montant"] > 100]
choix = ["petite", "grosse"]
commandes["taille"] = np.select(conditions, choix, default="pile")
commandes[["montant", "gros_panier", "taille"]].head()

### Texte : concaténation vectorisée

Dans la correction de l'exercice API, l'URL de chaque commune est construite par concaténation de colonnes, sans boucle :

In [ ]:
communes = pd.read_csv("../donnees/api/communes_exemple.csv")
base = "https://api.open-meteo.com/v1/forecast?"
communes["url"] = (base + "latitude=" + communes["lat"].astype(str)
                   + "&longitude=" + communes["lon"].astype(str) + "&hourly=temperature_2m")
communes["url"].iloc[0]

## 5. Mesurer l'écart

> **Comparaison.** Sur un million de valeurs, quel facteur de vitesse attendez-vous entre `apply` et la version vectorisée ?

In [ ]:
grand = pd.Series(np.random.default_rng(1).uniform(0, 10, 1_000_000))
%timeit grand.apply(taille, seuil=5)
%timeit (-1 * (grand < 5) + (grand > 5))
%timeit np.select([grand < 5, grand > 5], ["petite", "grosse"], default="pile")

L'écart dépasse souvent un facteur 100. Sur quelques centaines de lignes, la différence est imperceptible ; sur des millions de lignes ou dans une boucle d'entraînement, elle se compte en minutes.

## 6. Quand utiliser quoi ?

| Besoin | Outil |
|---|---|
| Remplacer des codes par des libellés | `Series.map(dict)` |
| Calcul arithmétique, comparaison | opérateurs vectorisés (`+`, `*`, `>`…) |
| Condition à 2 branches | `np.where` |
| Conditions multiples | `np.select`, ou `pd.cut` pour des tranches numériques |
| Texte | `.str` |
| Dates | `.dt` |
| Logique complexe, appel à une bibliothèque externe, fonction non vectorisable | `apply` |
| Une fonction par cellule | `DataFrame.map` |

`apply` reste légitime quand aucune opération vectorisée n'existe : interroger une API pour chaque ligne, parser un texte libre complexe, appeler un modèle. On l'utilise alors en connaissance de cause.

## Pièges fréquents

| Situation | Symptôme | Réflexe |
|---|---|---|
| Fonction appliquée sur des `NaN` | valeurs manquantes transformées en résultats faux | `na_action="ignore"` ou traiter `pd.isna` |
| `applymap` | `AttributeError` en Pandas 3 | `DataFrame.map` |
| `apply(axis=1)` sur une grande table | lenteur extrême | opérations vectorisées sur les colonnes |
| `map(dict)` avec une valeur absente | `NaN` inattendu | vérifier `isna().sum()` après le `map` |
| Fonction qui modifie une variable extérieure | résultats imprévisibles | une fonction `apply` doit seulement renvoyer une valeur |

## Exercices

- Exercice supplémentaire **S10** : réécrire un `apply` en version vectorisée et mesurer.
- Renforcement : drill **D04 Students Alcohol Consumption** (la solution de l'étape 10 utilise `applymap`, supprimé en Pandas 3).

## À retenir

- `map` : correspondance (dict, Series) ou fonction, valeur par valeur.
- `apply` : fonction avec arguments, par colonne ou par ligne (`axis=1`), éventuellement plusieurs colonnes en sortie.
- `DataFrame.map` remplace `applymap`.
- Attention aux `NaN` dans les fonctions personnelles.
- Vectoriser dès que possible : opérateurs, `np.where`, `np.select`, `.str`, `.dt`.